# Complete V1 Network Forward Pass

Run a full simulation from visual stimulus to spiking output.

## Table of Contents
1. [Pipeline Overview](#1-pipeline-overview)
2. [Building V1Network](#2-building-v1network)
3. [Initializing State](#3-initializing-state)
4. [Preparing Input](#4-preparing-input)
5. [Forward Pass](#5-forward-pass)
6. [Visualizing Results](#6-visualizing-results)
7. [JIT Compilation](#7-jit-compilation)
8. [Multi-Batch Inference](#8-multi-batch-inference)
9. [State Continuation](#9-state-continuation)

---

In [ ]:
# Setup
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import time

# Enable high-DPI display for plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

In [ ]:
# Configuration - MODIFY THESE PATHS
DATA_DIR = Path("/path/to/GLIF_network")  # <-- CHANGE THIS
LGN_DATA_PATH = Path("/path/to/lgn_full_col_cells_3.csv")  # <-- CHANGE THIS

# Verify data
files_found = {
    "GLIF_network": DATA_DIR.exists(),
    "LGN data": LGN_DATA_PATH.exists(),
}

for name, found in files_found.items():
    status = "Found" if found else "MISSING"
    print(f"{name}: {status}")

if not all(files_found.values()):
    print("\nPlease update DATA_DIR and LGN_DATA_PATH.")

## 1. Pipeline Overview

The complete V1 forward pass pipeline:

```
Visual Stimulus (movie)
       |
       v
   LGN Model
   - Spatial filtering (Gaussian)
   - Temporal filtering (kernels)
       |
       v
  LGN Firing Rates
       |
       v
   Input Layer
   - Sparse LGN->V1 connectivity
   - Background noise
       |
       v
  GLIF3 Network
   - Membrane dynamics
   - Spike generation
   - Recurrent connectivity
   - Adaptive currents
       |
       v
 V1 Spikes & Voltages
```

### Key Components

| Component | Description | Shape |
|-----------|-------------|-------|
| Movie | Visual stimulus | (T, H, W) |
| LGN rates | Firing rates | (T, n_lgn) |
| Input current | After sparse matmul | (T, batch, n_neurons*n_rec) |
| V1 spikes | Output spikes | (T, batch, n_neurons) |
| V1 voltages | Membrane potential | (T, batch, n_neurons) |

## 2. Building V1Network

The `V1Network` class integrates all components.

In [ ]:
# For demonstration without full network, let's use a simplified approach
# with direct GLIF3 simulation

try:
    from v1_jax.data import load_billeh
    from v1_jax.nn.glif3_cell import GLIF3Cell, GLIF3Params, glif3_step, glif3_unroll
    from v1_jax.lgn import LGN
    from v1_jax.data.stim_generator import make_drifting_grating, firing_rates_to_input
    from jax.experimental.sparse import BCOO
    
    # Load network subset
    print("Loading network...")
    n_neurons_subset = 5000  # Use subset for demo
    
    input_pop, network, bkg_weights = load_billeh(
        n_input=17400,
        n_neurons=n_neurons_subset,
        core_only=True,
        data_dir=str(DATA_DIR),
        seed=3000,
    )
    
    print(f"Network loaded: {network['n_nodes']:,} neurons, {network['n_edges']:,} synapses")
    
    # Create GLIF3 parameters from network
    print("\nCreating GLIF3 parameters...")
    glif3_params, metadata = GLIF3Cell.from_network(
        network,
        dt=1.0,
        gauss_std=0.5,
        dampening_factor=0.3,
        max_delay=5,
    )
    
    n_neurons = metadata['n_neurons']
    n_receptors = metadata['n_receptors']
    max_delay = metadata['max_delay']
    
    print(f"GLIF3 parameters created:")
    print(f"  n_neurons: {n_neurons:,}")
    print(f"  n_receptors: {n_receptors}")
    print(f"  max_delay: {max_delay}")
    
    NETWORK_LOADED = True
    
except FileNotFoundError as e:
    print(f"Error loading network: {e}")
    print("Using synthetic parameters for demonstration...")
    NETWORK_LOADED = False
    
    # Create synthetic parameters for demo
    n_neurons = 1000
    n_receptors = 4
    max_delay = 5

In [ ]:
if not NETWORK_LOADED:
    # Create synthetic GLIF3 parameters for demo
    from v1_jax.nn.glif3_cell import GLIF3Params
    
    # Typical cortical neuron parameters
    tau_m = 20.0  # ms
    decay = jnp.exp(-1.0 / tau_m)
    current_factor = (1 / tau_m) * (1 - jnp.exp(-1.0 / tau_m)) * tau_m
    
    tau_syn = jnp.array([2.0, 100.0, 6.0, 150.0])  # AMPA, NMDA, GABA_A, GABA_B
    syn_decay = jnp.exp(-1.0 / tau_syn)
    psc_initial = jnp.e / tau_syn
    
    glif3_params = GLIF3Params(
        v_reset=jnp.zeros(n_neurons),
        v_th=jnp.ones(n_neurons),
        e_l=jnp.zeros(n_neurons),
        t_ref=jnp.ones(n_neurons) * 2.0,
        decay=jnp.ones(n_neurons) * decay,
        current_factor=jnp.ones(n_neurons) * current_factor,
        syn_decay=jnp.tile(syn_decay, (n_neurons, 1)),
        psc_initial=jnp.tile(psc_initial, (n_neurons, 1)),
        param_k=jnp.ones((n_neurons, 2)) * 0.05,
        asc_amps=jnp.ones((n_neurons, 2)) * -0.3,
        param_g=jnp.ones(n_neurons) / tau_m,
        voltage_scale=jnp.ones(n_neurons) * 20.0,
        voltage_offset=jnp.ones(n_neurons) * -70.0,
    )
    
    print(f"Synthetic GLIF3 parameters created for {n_neurons} neurons")

## 3. Initializing State

Initialize the network state before simulation.

In [ ]:
from v1_jax.nn.glif3_cell import GLIF3Cell, GLIF3State

batch_size = 4

# Initialize state (zeros)
state = GLIF3Cell.init_state(
    n_neurons=n_neurons,
    n_receptors=n_receptors,
    max_delay=max_delay,
    batch_size=batch_size,
    params=glif3_params,
)

print("Initial state shapes:")
for field, value in zip(state._fields, state):
    print(f"  {field:12s}: {value.shape}")

print(f"\nInitial membrane potential (first 5 neurons, first batch):")
print(f"  {state.v[0, :5]}")

In [ ]:
# Random initialization (for faster convergence to steady-state)
key = jax.random.PRNGKey(42)

random_state = GLIF3Cell.random_state(
    n_neurons=n_neurons,
    n_receptors=n_receptors,
    max_delay=max_delay,
    batch_size=batch_size,
    params=glif3_params,
    key=key,
)

print("Random initial membrane potential (first 5 neurons, first batch):")
print(f"  {random_state.v[0, :5]}")

## 4. Preparing Input

Generate input stimulus and convert to GLIF3 input current.

In [ ]:
from v1_jax.data.stim_generator import make_drifting_grating

# Generate drifting grating stimulus
key = jax.random.PRNGKey(0)
seq_len = 100  # ms

grating = make_drifting_grating(
    row_size=120,
    col_size=240,
    moving_flag=True,
    image_duration=seq_len,
    theta=45.0,  # Orientation
    cpd=0.05,
    temporal_f=2.0,
    contrast=1.0,
    key=key,
)

print(f"Grating stimulus shape: {grating.shape}")

# Visualize first and last frame
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(grating[0, :, :, 0], cmap='gray', vmin=-1, vmax=1)
axes[0].set_title('t=0ms')
axes[0].axis('off')

axes[1].imshow(grating[-1, :, :, 0], cmap='gray', vmin=-1, vmax=1)
axes[1].set_title(f't={seq_len-1}ms')
axes[1].axis('off')

plt.suptitle('Drifting Grating Stimulus', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# For simplified demo, create synthetic input current
# In full pipeline, this would come from LGN model

# Create random input current (simulating LGN input)
key = jax.random.PRNGKey(1)
input_amplitude = 2.0

# Shape: (seq_len, batch, n_neurons * n_receptors)
inputs = jax.random.normal(key, (seq_len, batch_size, n_neurons * n_receptors))
inputs = inputs * input_amplitude / n_receptors

# Add temporal structure (sinusoidal modulation)
time_mod = jnp.sin(2 * jnp.pi * jnp.arange(seq_len) / 50)[:, None, None]
inputs = inputs * (0.5 + 0.5 * time_mod)

print(f"Input shape: {inputs.shape}")
print(f"Input range: [{float(inputs.min()):.2f}, {float(inputs.max()):.2f}]")

## 5. Forward Pass

Run the network simulation using `glif3_unroll`.

In [ ]:
from v1_jax.nn.glif3_cell import glif3_unroll

# For simplified demo, use identity recurrent function (no recurrent connections)
# In full network, this would be sparse matmul with recurrent weights
def no_recurrent_fn(z_buf):
    return jnp.zeros((batch_size, n_neurons * n_receptors))

print("Running forward pass...")
start_time = time.time()

final_state, all_spikes, all_voltages = glif3_unroll(
    params=glif3_params,
    initial_state=state,
    inputs=inputs,
    recurrent_fn=no_recurrent_fn,
    n_neurons=n_neurons,
    n_receptors=n_receptors,
    max_delay=max_delay,
    dt=1.0,
    gauss_std=0.5,
    dampening_factor=0.3,
)

# Block until computation is done
jax.block_until_ready(all_spikes)
elapsed = time.time() - start_time

print(f"\nForward pass completed in {elapsed*1000:.1f}ms")
print(f"\nOutput shapes:")
print(f"  Spikes:   {all_spikes.shape}")
print(f"  Voltages: {all_voltages.shape}")

In [ ]:
# Spike statistics
total_spikes = int(jnp.sum(all_spikes))
spikes_per_neuron = total_spikes / (n_neurons * batch_size * seq_len) * 1000

print("Spike Statistics:")
print(f"  Total spikes: {total_spikes:,}")
print(f"  Mean firing rate: {spikes_per_neuron:.1f} Hz")
print(f"  Spikes per timestep: {total_spikes / seq_len:.1f}")

# Per-batch statistics
for b in range(batch_size):
    batch_spikes = int(jnp.sum(all_spikes[:, b, :]))
    batch_rate = batch_spikes / (n_neurons * seq_len) * 1000
    print(f"  Batch {b}: {batch_spikes:,} spikes ({batch_rate:.1f} Hz)")

## 6. Visualizing Results

Visualize spike rasters, firing rates, and membrane potentials.

In [ ]:
# Spike raster plot (sample neurons from first batch)
n_sample = min(100, n_neurons)
key = jax.random.PRNGKey(42)
sample_neurons = jax.random.choice(key, n_neurons, (n_sample,), replace=False)
sample_neurons = jnp.sort(sample_neurons)

sample_spikes = all_spikes[:, 0, sample_neurons]  # First batch

# Find spike times
spike_times = []
spike_neurons = []
for i, neuron_idx in enumerate(range(n_sample)):
    times = jnp.where(sample_spikes[:, neuron_idx] > 0.5)[0]
    spike_times.extend(times.tolist())
    spike_neurons.extend([i] * len(times))

plt.figure(figsize=(12, 6))
plt.scatter(spike_times, spike_neurons, s=1, c='black', alpha=0.7)
plt.xlabel('Time (ms)')
plt.ylabel('Neuron (sampled)')
plt.title(f'Spike Raster Plot ({n_sample} sampled neurons, batch 0)')
plt.xlim([0, seq_len])
plt.ylim([-1, n_sample])
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Population firing rate over time
pop_rate = jnp.mean(all_spikes[:, 0, :], axis=1) * 1000  # Hz (assuming 1ms bins)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Input modulation
input_mean = jnp.mean(inputs[:, 0, :], axis=1)
axes[0].plot(input_mean, 'b-', linewidth=1.5)
axes[0].set_ylabel('Mean Input')
axes[0].set_title('Input Current')
axes[0].grid(True, alpha=0.3)

# Population rate
axes[1].plot(pop_rate, 'r-', linewidth=1.5)
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Population Rate (Hz)')
axes[1].set_title('Mean Population Firing Rate')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Membrane potential traces
n_traces = 5
trace_neurons = sample_neurons[:n_traces]

fig, axes = plt.subplots(n_traces, 1, figsize=(12, 10), sharex=True)

for i, neuron_idx in enumerate(trace_neurons):
    voltage_trace = all_voltages[:, 0, neuron_idx]  # First batch
    spike_times = jnp.where(all_spikes[:, 0, neuron_idx] > 0.5)[0]
    
    axes[i].plot(voltage_trace, 'b-', linewidth=1)
    
    # Mark spikes
    if len(spike_times) > 0:
        axes[i].scatter(spike_times, [float(voltage_trace[t]) for t in spike_times],
                       c='red', s=50, zorder=5, label='Spikes')
    
    # Threshold line
    v_th = float(glif3_params.v_th[neuron_idx] * glif3_params.voltage_scale[neuron_idx] + 
                 glif3_params.voltage_offset[neuron_idx])
    axes[i].axhline(y=v_th, color='gray', linestyle='--', alpha=0.5)
    
    axes[i].set_ylabel(f'Neuron {int(neuron_idx)}\n(mV)')
    axes[i].grid(True, alpha=0.3)
    
    if i == 0:
        axes[i].legend(loc='upper right')

axes[-1].set_xlabel('Time (ms)')
plt.suptitle('Membrane Potential Traces', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Spike activity heatmap
subsample = min(500, n_neurons)
neuron_subset = jnp.arange(0, n_neurons, n_neurons // subsample)[:subsample]

plt.figure(figsize=(12, 6))
plt.imshow(
    all_spikes[:, 0, neuron_subset].T,
    aspect='auto',
    cmap='binary',
    interpolation='nearest',
    extent=[0, seq_len, 0, subsample],
)
plt.colorbar(label='Spike')
plt.xlabel('Time (ms)')
plt.ylabel('Neuron Index (subsampled)')
plt.title(f'Spike Activity Heatmap ({subsample} neurons)')
plt.show()

## 7. JIT Compilation

JIT compilation dramatically speeds up the forward pass.

In [ ]:
# Create JIT-compiled forward function
@jax.jit
def forward_pass(params, state, inputs):
    """JIT-compiled forward pass."""
    def no_recurrent(z_buf):
        return jnp.zeros((batch_size, n_neurons * n_receptors))
    
    return glif3_unroll(
        params=params,
        initial_state=state,
        inputs=inputs,
        recurrent_fn=no_recurrent,
        n_neurons=n_neurons,
        n_receptors=n_receptors,
        max_delay=max_delay,
        dt=1.0,
        gauss_std=0.5,
        dampening_factor=0.3,
    )

# First call: includes compilation time
print("First call (includes JIT compilation)...")
start = time.time()
_, spikes, _ = forward_pass(glif3_params, state, inputs)
jax.block_until_ready(spikes)
first_call_time = time.time() - start
print(f"  Time: {first_call_time*1000:.1f}ms")

# Second call: uses compiled version
print("\nSecond call (JIT compiled)...")
start = time.time()
_, spikes, _ = forward_pass(glif3_params, state, inputs)
jax.block_until_ready(spikes)
second_call_time = time.time() - start
print(f"  Time: {second_call_time*1000:.2f}ms")

print(f"\nSpeedup: {first_call_time/second_call_time:.0f}x")

In [ ]:
# Benchmark multiple runs
n_runs = 10
times = []

for _ in range(n_runs):
    start = time.time()
    _, spikes, _ = forward_pass(glif3_params, state, inputs)
    jax.block_until_ready(spikes)
    times.append(time.time() - start)

mean_time = np.mean(times) * 1000
std_time = np.std(times) * 1000

print(f"Benchmark ({n_runs} runs):")
print(f"  Mean time: {mean_time:.2f} ms")
print(f"  Std:       {std_time:.2f} ms")
print(f"  Throughput: {seq_len / mean_time * 1000:.0f} simulated ms/second")

## 8. Multi-Batch Inference

Process multiple inputs in parallel using batching.

In [ ]:
# Test different batch sizes
batch_sizes = [1, 4, 8, 16]
throughputs = []

for bs in batch_sizes:
    # Create state and inputs for this batch size
    test_state = GLIF3Cell.init_state(
        n_neurons=n_neurons,
        n_receptors=n_receptors,
        max_delay=max_delay,
        batch_size=bs,
        params=glif3_params,
    )
    
    test_inputs = jax.random.normal(
        jax.random.PRNGKey(0),
        (seq_len, bs, n_neurons * n_receptors)
    ) * input_amplitude / n_receptors
    
    # Redefine forward for this batch size
    @jax.jit
    def forward_bs(state, inputs):
        def no_recurrent(z_buf):
            return jnp.zeros((bs, n_neurons * n_receptors))
        return glif3_unroll(
            glif3_params, state, inputs, no_recurrent,
            n_neurons, n_receptors, max_delay, 1.0, 0.5, 0.3
        )
    
    # Warmup
    _ = forward_bs(test_state, test_inputs)
    
    # Benchmark
    times = []
    for _ in range(5):
        start = time.time()
        _, spikes, _ = forward_bs(test_state, test_inputs)
        jax.block_until_ready(spikes)
        times.append(time.time() - start)
    
    mean_time = np.mean(times)
    throughput = bs * seq_len / mean_time  # samples * timesteps per second
    throughputs.append(throughput)
    
    print(f"Batch size {bs:2d}: {mean_time*1000:.2f}ms, throughput={throughput/1000:.1f}k timesteps/s")

# Plot throughput scaling
plt.figure(figsize=(8, 5))
plt.bar(range(len(batch_sizes)), [t/1000 for t in throughputs], tick_label=batch_sizes)
plt.xlabel('Batch Size')
plt.ylabel('Throughput (k timesteps/s)')
plt.title('Throughput Scaling with Batch Size')
plt.grid(True, alpha=0.3, axis='y')
plt.show()

## 9. State Continuation

Continue simulation from a previous state for multi-segment processing.

In [ ]:
# Segment 1: Run first 50ms
segment1_inputs = inputs[:50]

final_state1, spikes1, voltages1 = forward_pass(glif3_params, state, segment1_inputs)
print(f"Segment 1: {segment1_inputs.shape[0]}ms")
print(f"  Spikes: {int(jnp.sum(spikes1))}")

# Segment 2: Continue from final_state1
segment2_inputs = inputs[50:]

final_state2, spikes2, voltages2 = forward_pass(glif3_params, final_state1, segment2_inputs)
print(f"\nSegment 2: {segment2_inputs.shape[0]}ms")
print(f"  Spikes: {int(jnp.sum(spikes2))}")

# Combined
combined_spikes = jnp.concatenate([spikes1, spikes2], axis=0)
combined_voltages = jnp.concatenate([voltages1, voltages2], axis=0)

print(f"\nCombined: {combined_spikes.shape[0]}ms")
print(f"  Total spikes: {int(jnp.sum(combined_spikes))}")

In [ ]:
# Verify continuity: compare to full run
_, full_spikes, full_voltages = forward_pass(glif3_params, state, inputs)

# Check if combined matches full
spike_match = jnp.allclose(combined_spikes, full_spikes)
voltage_match = jnp.allclose(combined_voltages, full_voltages)

print("State Continuation Verification:")
print(f"  Spikes match: {spike_match}")
print(f"  Voltages match: {voltage_match}")

if spike_match and voltage_match:
    print("\nState continuation is working correctly!")

---

## Summary

Key takeaways:

1. **Pipeline**: Movie -> LGN -> Input Layer -> GLIF3 -> Spikes

2. **V1Network**: Integrates all components with `from_billeh()` factory method

3. **State Management**: Use `init_state()` or `random_state()` to initialize

4. **Forward Pass**: `glif3_unroll()` uses `lax.scan` for efficient time unrolling

5. **JIT Compilation**: Provides 100x+ speedup after first call

6. **Batching**: Process multiple inputs in parallel for efficiency

7. **State Continuation**: Pass final state to continue simulation

---

## Exercises

1. **Add recurrent connections**: Create sparse recurrent weights and integrate with `recurrent_fn`.

2. **Try different stimuli**: Compare responses to different grating orientations.

3. **Analyze tuning**: Measure orientation selectivity of individual neurons.

In [ ]:
# Exercise starter: Orientation tuning curve

orientations = [0, 45, 90, 135, 180]

print("TODO: Implement orientation tuning analysis")
print("Hint: Run simulation for each orientation and measure mean firing rate")

---

## Source Files

- `src/v1_jax/models/v1_network.py` - V1Network class
- `src/v1_jax/nn/glif3_cell.py` - GLIF3 neuron implementation
- `src/v1_jax/nn/sparse_layer.py` - Sparse connectivity
- `src/v1_jax/lgn/lgn_model.py` - LGN preprocessing

## References

- Chen et al., "Data-driven models of visual cortex", Science Advances 2022
- JAX Documentation: https://jax.readthedocs.io/